In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import kagglehub

from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

import warnings
warnings.filterwarnings('ignore')



In [ ]:

import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
# Load the dataset
r_path = os.path.join(path, 'Q1_data.csv')
df_r = pd.read_csv(r_path)

print(f"Dataset shape: {df_r.shape}")


In [ ]:
# Task 2: Write your code here:
df_r.head()

In [ ]:
# Task 3: Write your code here:
df_r.info()

In [ ]:
missing_percentage = (df_r.isnull().sum() / len(df_r)) * 100
missing_data = pd.DataFrame({
    'Column': missing_percentage.index,
    'Missing_Percentage': missing_percentage.values
})
missing_data = missing_data[missing_data['Missing_Percentage'] > 0].sort_values('Missing_Percentage', ascending=False)

print("Missing Data Analysis:")
missing_data.head(10)

In [ ]:
# Task 4: Write your code here:
df_r.describe()


In [ ]:
# Task 5: Write your code here:plt.figure(figsize=(10, 5))
plt.hist(df_r['Delivery_Time'].dropna(), bins=50, edgecolor='black')
plt.title('Delivery_Time')
plt.xlabel('Delivery_Time')
plt.ylabel('Traffic_Level')
plt.show()

In [ ]:
# Task 1: Write your code here:
cols = ['Order_ID', 'Delivery_Time', 'Traffic_Level', 'Weather', 'Time_of_Day', 'Distance_km', 'Preparation_Time_min']
df_clean = df_r[cols].copy()


print(f"Before: {df_clean.shape}")
df_clean = df_clean.dropna(subset=['Order_ID'])
print(f"After dropping missing Delivery_Time/Weather/Traffic_Level/Time_of_Day/Courier_Experience_yrs: {df_clean.shape}")




In [ ]:
# Task 2: Write your code here:
#df_clean[col] = df_clean[col].fillna('unknown')


df_clean['Order_ID'] = df_clean['Order_ID'].fillna(df_clean['Order_ID'].mode()[0])

print("Missing values remaining:", df_clean.isnull().sum().sum())

In [ ]:
# Task 3: Write your code here:
def check_duplicates(df):
  duplicates = df_r.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df_r.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df_r)

In [ ]:
# Task 4: Write your code here:
categorical_cols = ['Order_ID', 'Distance_km', 'Traffic_Level', 'Delivery_Time']
for col in categorical_cols:
    le = LabelEncoder()
    df_clean[col] = le.fit_transform(df_clean[col].astype(str))

df_clean.head()

In [ ]:
# Task 5: Write your code here:
data_for_scale = pd.DataFrame({"Feature_1": [471, 710, 714,625],"Feature_2": [50, 92, 66,44]})

print("Before scaling:")
data_for_scale



In [ ]:
# Task 6: Write your code here:

def mean_squared_error(y, y_hat):
  return (1 / (2 * len(y))) * np.sum((y_hat - y) ** 2)


In [ ]:
def gradient_descent(X, y, learning_rate, n_iters=500):
  m, n = X.shape  # m rows, n columns (dimensions)
  theta = np.zeros(n)  # initialize a zeros weight vector with n dimensions
  losses = []

  for _ in tqdm(range(n_iters), desc="Training Linear Regression"):
    y_hat = np.dot(X, theta)
    gradient = np.dot(X.T, (y_hat - y)) / m
    theta -= learning_rate * gradient

    loss = mean_squared_error(y, y_hat)
    losses.append(loss)

  return theta, losses

In [ ]:
# Task 1: Write your code here:
feature_cols = ['Order_ID', 'Delivery_Time', 'Traffic_Level', 'Weather', 'Time_of_Day', 'Distance_km', 'Preparation_Time_min']
X = df_clean[feature_cols]
y = df_clean['Delivery_Time']

# Train-test split (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")


In [ ]:
# Task 2,3,4,5: Write your code here:




    # showing class distribution
    train_ratio = (y_train.value_counts(normalize=True) * 100).sort_index()
    test_ratio = (y_test.value_counts(normalize=True) * 100).sort_index()

    print("  y_train class percentages:", {k: f"{v:.2f}%" for k, v in train_ratio.items()})
    print("  y_test class percentages :", {k: f"{v:.2f}%" for k, v in test_ratio.items()})

    print("-" * 40)




y_pred = model.predict(X_test_scaled)

mae = mean_absolute_error(y_test, y_pred)
model.fit(X_train, y_train)
print(f"MAE:  ${mae:,.2f}")




# Train Random Forest Regressor
model = RandomForestRegressor(n_estimators=100, max_depth=20, random_state=42, n_jobs=-1)
model.fit(X_train_scaled, y_train)
print("Model trained!")







In [ ]:
# Train Random Forest Regressor
model = RandomForestRegressor(n_estimators=100, max_depth=20, random_state=42, n_jobs=-1)
model.fit(X_train_scaled, y_train)
print("Model trained!")

In [ ]:
kfold = KFold(n_splits=5, shuffle=True, random_state=42)

mae_scores = []
rmse_scores = []

for train_idx, val_idx in kfold.split(X_train_scaled):
    X_fold_train, X_fold_val = X_train_scaled[train_idx], X_train_scaled[val_idx]
    y_fold_train, y_fold_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

    # Train and predict
    model.fit(X_fold_train, y_fold_train)
    y_fold_pred = model.predict(X_fold_val)

    # Calculate metrics
    mae_scores.append(mean_absolute_error(y_fold_val, y_fold_pred))
    rmse_scores.append(np.sqrt(mean_squared_error(y_fold_val, y_fold_pred)))

mae_scores = np.array(mae_scores)
rmse_scores = np.array(rmse_scores)

print(f"5-Fold CV Results:")
print(f"MAE:  ${mae_scores.mean():,.2f}")
print(f"RMSE: ${rmse_scores.mean():,.2f}")

In [ ]:
from sklearn.model_selection import KFold

# Use previously generated random data (example: regression data)
X, y = X_reg.copy(), y_reg.copy()

# Define K-Fold Cross Validation
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# Iterate through folds
for fold, (train_idx, test_idx) in enumerate(kf.split(X), start=1):
    # indexing for each fold
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    # print shapes
    print(f"Fold {fold}")
    print("  X_train shape:", X_train.shape)
    print("  X_test shape :", X_test.shape)
    print("  y_train shape:", y_train.shape)
    print("  y_test shape :", y_test.shape)
    print("-" * 30)

In [ ]:
# Task 1: Write your code here:

print("\nDataset Shapes")
print("X:", X.shape)
print("y:", y.shape)

# Encode Categorical Features
label_encoder = LabelEncoder()

for col in X.select_dtypes(include=["object"]).columns:
    X[col] = label_encoder.fit_transform(X[col])

# Feature Scaling
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


# K-Fold Cross Validation
kf = KFold(n_splits=5, shuffle=True, random_state=42)

mse_scores = []
mae_scores = []

for train_idx, test_idx in kf.split(X_scaled):
    X_train, X_test = X_scaled[train_idx], X_scaled[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    # Train model

    # Predict




# Plot Predictions vs Ground Truth
plt.figure(figsize=(6, 4))
plt.scatter(y_test, y_pred, alpha=0.6)
plt.plot(
    [y_test.min(), y_test.max()],
    [y_test.min(), y_test.max()],
    "r--",
    linewidth=2
)

plt.xlabel("Actual Exam Scores (Ground Truth)")
plt.ylabel("Predicted Exam Scores")
plt.title("Linear Regression: Predictions vs Ground Truth")
plt.grid(True)

plt.tight_layout()
plt.show()



In [ ]:
# Task 2: Write your code here:


In [ ]:
# Price distribution (target variable)
plt.figure(figsize=(10, 5))
plt.hist(df_r['Delivery_Time'].dropna(), bins=50, edgecolor='black')
plt.title('Delivery_Time')
plt.xlabel('Delivery_Time')
plt.ylabel('Distance_km')
plt.show()

In [ ]:
# Task Bonus: Write your code here: